# 自建GPT训练流程 Part 4: DPO 偏好对齐 教案

**课程名称：** DPO 偏好对齐——让自建模型学会「要点式」回答

**预计总时长：** 70-80 分钟

**源文件：** `Custom_GPT_Training/04_DPO_Training.ipynb`（共 22 个 Cell，Cell 0-21）

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 0-10 min | 开场 + DPO 原理 + RLHF 对比 | Cell 0-1 | 10 min |
| 10-15 min | 环境设置 | Cell 2-3 | 5 min |
| 15-27 min | 偏好数据加载与分析 | Cell 4-5 | 12 min |
| 27-40 min | DPO 数据集实现 | Cell 6-9 | 13 min |
| 40-45 min | 休息 + 回顾 | — | 5 min |
| 45-60 min | DPO Trainer 核心实现 | Cell 10-11 | 15 min |
| 60-70 min | 执行训练 + 可视化 | Cell 12-15 | 10 min |
| 70-78 min | SFT vs DPO 对比测试 | Cell 16-20 | 8 min |
| 78-80 min | 总结 + 下一步 | Cell 21 | 2 min |

---

## 课前准备

- [ ] 确认已完成 Part 3 SFT 训练，`models/custom_gpt/sft_model` 目录存在
- [ ] 确认 `models/custom_gpt/tokenizer.pkl` 存在（词表大小 381）
- [ ] 确认 `data/custom_dpo_train.jsonl`、`custom_dpo_val.jsonl`、`custom_dpo_test.jsonl` 已就绪
- [ ] `torch`, `numpy`, `matplotlib`, `tqdm` 已安装
- [ ] GPU 优先；CPU 可跑但训练更慢
- [ ] 提前运行一遍 notebook 确认输出正常
- [ ] 准备白板——DPO 损失函数公式需要板书辅助

---

## 第一段：开场 + DPO 原理 + RLHF 对比（Cell 0-1）

📍 浏览 Cell 0（标题、DPO vs RLHF 流程图、DPO 核心思想与公式）、Cell 1（学习路线表 + 本步骤目标）

⏱ 时间分配：10 分钟

🎯 本段目标
- 回顾 SFT 的局限性，引出偏好对齐的动机
- 通过 Cell 0 的流程图对比 RLHF（4步）与 DPO（2步）的差异
- 建立对 DPO 损失函数的初步直觉：增加 chosen 概率、降低 rejected 概率

🗣 讲课话术

> 大家好！上一节我们做完了 SFT 指令微调，模型已经学会了「能回答问题」。但是光能回答还不够——同一个问题可以有很多种回答方式，有的简洁有条理，有的啰嗦没重点。今天我们要用 DPO 教模型「怎么回答更好」。
>
> 来看 Cell 0 的流程对比。传统 RLHF 要走四步——SFT、训练 Reward Model、PPO 强化学习、最后才得到对齐模型。Cell 0 说得很直白：「复杂、不稳定、计算量大」。而 DPO 呢？只要两步——SFT 模型加偏好数据直接优化。为什么能省掉 Reward Model？因为 DPO 的公式把奖励模型「消化」进了损失函数里。
>
> 看 Cell 0 底部的公式。核心思想很简单——给同一个 prompt 准备两个回答：chosen（人类喜欢的）和 rejected（人类不喜欢的）。DPO 的目标就是让模型给 chosen 更高的概率，给 rejected 更低的概率。公式里的 π 是当前模型，π_ref 是参考模型（就是 SFT 模型），β 控制偏离程度。
>
> 打个比方：参考模型就像你的「原始性格」，DPO 训练就像一个教练在教你「这种回答方式比那种好」。β 就是教练的严格程度——β 太小，你变化太大可能变得不像自己；β 太大，几乎没变化，白训了。
>
> Cell 1 列出了本步骤的四个目标：构建偏好数据集、实现 DPO 训练器、执行训练、验证效果。预计用时：阅读 15 分钟、训练 10 分钟、对比 10 分钟。

👀 输出要点
- Cell 0：RLHF = 4 步（复杂），DPO = 2 步（简单）
- Cell 0 公式：Loss = -log(σ(β * (log_ratio_chosen - log_ratio_rejected)))
- Cell 1：四个学习目标，前置知识为 Part 3 SFT + Ch10 DPO 原理

❓ 预判问题

Q: DPO 比 RLHF 简单这么多，效果会差吗？
A: 论文实验显示 DPO 效果接近甚至超过 RLHF。Llama 3、Zephyr 等模型都用了 DPO 或其变体。DPO 的优势在于实现简单、训练稳定，缺点是依赖离线偏好数据质量。

Q: 这里的 π_ref 为什么要用 SFT 模型？
A: 参考模型是 KL 散度约束的锚点，防止训练后的模型偏离太远。用 SFT 模型作为起点最自然——它已经学会了回答问题的基本能力，DPO 只需要在此基础上调整「回答风格」。

➡️ 转场

> 好，原理框架建立了。我们先把环境跑起来。

---

## 第二段：环境设置（Cell 2-3）

📍 浏览 Cell 2（Markdown 标题）、运行 Cell 3（导入依赖、设备检测、随机种子）

⏱ 时间分配：5 分钟

🎯 本段目标
- 确认运行环境就绪
- 检测 GPU/CPU 设备
- 加载 custom_gpt 模块（CustomGPT、GPTConfig、SimpleTokenizer）

🗣 讲课话术

> 运行 Cell 3。（运行 Cell 3）看导入列表——torch、numpy、matplotlib 是老朋友了。注意这里还导入了 `copy` 模块，因为我们待会需要深拷贝 SFT 模型作为参考模型。
>
> 关键导入是 `from custom_gpt import CustomGPT, GPTConfig, SimpleTokenizer, count_parameters`——这些都是我们在 Part 1-3 一步步搭建的组件。DPO 训练是站在 SFT 的肩膀上的。
>
> 看输出——`使用设备: cuda`。如果你是 CPU 也没关系，代码会自动适配，就是训练那一步慢一些。随机种子固定在 42，确保结果可复现。

👀 输出要点
- Cell 3 输出：`使用设备: cuda`
- 随机种子：torch.manual_seed(42)、np.random.seed(42)

❓ 预判问题

Q: CPU 跑 DPO 训练要多久？
A: Cell 1 标注的预计是训练约 10 分钟（CPU）。我们后面会限制 max_steps=30，所以不会太久。

➡️ 转场

> 环境 OK。下面看偏好数据——DPO 的燃料。

---

## 第三段：偏好数据加载与分析（Cell 4-5）

📍 浏览 Cell 4（Markdown：数据路线说明）、运行 Cell 5（加载 JSONL 数据 + 输出统计与示例）

⏱ 时间分配：12 分钟

🎯 本段目标
- 理解偏好数据的三元组结构：(prompt, chosen, rejected)
- 理解本路线的偏好策略：结构化「要点」风格 vs 非结构化叙述
- 通过数据量和示例建立对数据规模的认知

🗣 讲课话术

> Cell 4 说了本路线的偏好策略——我们聚焦「短知识问答」，chosen 偏好结构化的「要点」风格（定义+作用），rejected 是非结构化的散文式回答。
>
> 运行 Cell 5。（运行 Cell 5）看数据量——训练集 **8000 条**，验证集 **1000 条**，测试集 **1000 条**。这个规模比 Ch10 的 30 条大得多，训练效果也会更明显。
>
> 看示例数据：
> - prompt: 「用简洁语言介绍Temperature」
> - chosen: 「要点：1) 定义：控制采样随机性的参数 2) 作用：数值越高越发散。」
> - rejected: 「Temperature是控制采样随机性的参数，主要用于数值越高越发散。」
> - category: 「知识问答」
>
> 注意区别——chosen 用了「要点：1) 定义：... 2) 作用：...」的结构化格式，rejected 只是普通的叙述句。两者信息量差不多，但组织方式不同。DPO 要学的就是这种格式偏好。
>
> 这跟 Ch10 的「简洁 vs 啰嗦」不一样——我们这里不是让回答更短，而是让回答更有结构。这是一种更精细的风格对齐。
>
> 互动提问：大家觉得这两种回答，哪种在实际工作中更有用？对，结构化的更好扫读、更容易提取关键信息。这就是我们希望模型学会的。

👀 输出要点
- Cell 5 输出：DPO训练集: 8000 条 | 验证集: 1000 条 | 测试集: 1000 条
- 示例三元组：prompt=「用简洁语言介绍Temperature」，chosen=「要点：...」结构化格式，rejected=叙述格式
- category: 知识问答

❓ 预判问题

Q: 8000 条偏好数据在实际中算多吗？
A: 中等规模。Anthropic 的 HH-RLHF 有 16 万条，UltraFeedback 有 6 万条。8000 条对于我们这个 14M 参数的小模型来说足够了。关键是数据质量——每一条的 chosen/rejected 对比要清晰一致。

Q: chosen 和 rejected 的信息量一样吗？
A: 有意设计成信息量接近——都包含定义和作用。区别在于组织方式：chosen 用「要点」结构化格式，rejected 用普通叙述。这样 DPO 学到的是格式偏好，而非信息量偏好。

Q: 数据文件为什么是 JSONL 格式？
A: JSONL 是每行一个 JSON 对象，方便流式读取和追加写入。大规模偏好数据集通常都用 JSONL 或 Parquet 格式。

➡️ 转场

> 数据有了，下面要把它变成模型能吃的 Dataset——包括 ChatML 格式化和 token 编码。

---

## 第四段：DPO 数据集实现（Cell 6-9）

📍 浏览 Cell 6（Markdown 标题）、运行 Cell 7（ChatMLFormatter 类）、运行 Cell 8（DPODataset 类）、运行 Cell 9（加载 tokenizer + 创建 DataLoader）

⏱ 时间分配：13 分钟

🎯 本段目标
- 理解 ChatML 格式在 DPO 中的作用：统一 prompt 格式
- 理解 DPODataset 的核心设计：返回 chosen_ids、rejected_ids、prompt_length
- 理解 prompt_length 的作用：只计算 response 部分的概率
- 确认 tokenizer 和 DataLoader 加载成功

🗣 讲课话术

> Cell 7 复用了 SFT 阶段的 ChatMLFormatter。两个关键方法：`format()` 生成完整对话（含回答），`format_prompt_only()` 只生成到 `<|assistant|>` 为止。DPO 训练时两个都要用——format 用于算概率，format_prompt_only 用于确定 prompt 长度。
>
> 运行 Cell 8 看 DPODataset。这个类的 `__getitem__` 返回 5 个东西：
> 1. `chosen_ids` — chosen 回答的完整 token 序列
> 2. `rejected_ids` — rejected 回答的完整 token 序列
> 3. `chosen_mask` — chosen 的 attention mask（区分真实 token 和 padding）
> 4. `rejected_mask` — rejected 的 attention mask
> 5. `prompt_length` — prompt 部分的长度
>
> 为什么需要 prompt_length？这是 DPO 实现中的一个**关键细节**。我们只想计算 response 部分的 log 概率——因为 prompt 对 chosen 和 rejected 是一样的，把 prompt 的概率算进去只会引入噪声。prompt_length 就是用来做这个截断的。
>
> 运行 Cell 9。（运行 Cell 9）看输出——加载了预训练 tokenizer，词表大小 **381**（字符级分词器）。训练集 **8000 样本**，验证集 **1000 样本**。BATCH_SIZE 设为 4，MAX_LENGTH 设为 256。
>
> 注意 Cell 9 的 fallback 逻辑——如果没有预训练 tokenizer，会用 DPO 数据现场构建一个。但正常流程应该有 Part 1-3 留下的 tokenizer。

👀 输出要点
- Cell 7：ChatMLFormatter 定义了 SYSTEM_TOKEN、USER_TOKEN、ASSISTANT_TOKEN、END_TOKEN
- Cell 8：DPODataset 返回 chosen_ids, rejected_ids, chosen_mask, rejected_mask, prompt_length
- Cell 9 输出：
  - 加载tokenizer, 词表大小: 381
  - DPO数据集: 8000 条（训练）、1000 条（验证）
  - BATCH_SIZE=4, MAX_LENGTH=256

❓ 预判问题

Q: 为什么 MAX_LENGTH 只有 256？
A: 我们的模型是 14M 参数的小模型，训练数据是短知识问答。256 个 token 足够覆盖 prompt + response。生产级 DPO 训练通常用 512-2048。

Q: 字符级分词器和 BPE 在 DPO 中有什么区别？
A: 字符级分词器每个字符是一个 token，序列更长但词表更小。BPE 序列更短但词表更大。对 DPO 训练本身没有本质影响——关键是 chosen 和 rejected 用同样的分词器。

Q: attention_mask 在这里起什么作用？
A: 区分真实 token（mask=1）和 padding（mask=0）。因为 chosen 和 rejected 长度不同，需要 padding 到 max_length。计算 log-prob 时 padding 位置不能算进去。

➡️ 转场

> 数据管道搭好了。我们休息一下，回顾前半段内容。

---

## 休息 + 回顾（第 40-45 分钟）

⏱ 时间分配：5 分钟

**三句话回顾前半段：**

1. DPO 是 RLHF 的简化版——不需要单独训练 Reward Model，把奖励模型「消化」进了损失函数。核心公式是 Loss = -log(σ(β * Δ))，Δ 是 chosen 与 rejected 的 log ratio 差。
2. 我们的偏好数据有 8000 条训练集（1000 验证、1000 测试），偏好策略是结构化「要点」风格 vs 非结构化叙述——信息量相近，组织方式不同。
3. DPODataset 的关键设计是记录 prompt_length，确保只计算 response 部分的概率，避免 prompt 部分引入噪声。

**下一段预告：** 接下来是全课最核心的部分——DPO Trainer 的实现。我们要看 log_probs 怎么算、DPO loss 怎么写、训练循环怎么跑。

---

## 第五段：DPO Trainer 核心实现（Cell 10-11）

📍 浏览 Cell 10（Markdown：DPO 训练器说明 + loss 公式）、运行 Cell 11（DPOTrainer 完整类定义）

⏱ 时间分配：15 分钟（初始化 3 分钟 + log_probs 5 分钟 + DPO loss 5 分钟 + 训练循环 2 分钟）

🎯 本段目标
- 理解 DPO 训练器的双模型架构：policy model（可训练）+ reference model（冻结）
- 深入理解 compute_log_probs：如何只统计 response 部分的 log 概率
- 掌握 compute_dpo_loss 的核心计算：4 步走完 DPO
- 理解 reward margin 作为训练监控指标的意义

🗣 讲课话术

> Cell 10 再次给出 DPO 的核心公式：Loss = -log(σ(β * (log_ratio_chosen - log_ratio_rejected)))。其中 log_ratio = log π(response) - log π_ref(response)。
>
> 运行 Cell 11。这个 DPOTrainer 类是本课的核心，我们逐块拆解。
>
> **第一块：初始化（__init__）**
> 两个模型——self.model 是 policy，会训练；self.ref_model 是参考，eval() 模式且所有参数 requires_grad=False。打个比方：policy 是学生，ref_model 是学生入学时的照片——我们希望学生进步，但不要变得面目全非。β=0.5 控制「允许变多少」。学习率 3e-6，比 SFT 的学习率小很多——DPO 是微调中的微调，步子不能太大。
>
> **第二块：compute_log_probs**（重点，在白板上画示意图）
> 1. 把 input_ids 送入模型，得到 logits
> 2. 对 logits 做 log_softmax，得到每个位置每个 token 的 log 概率
> 3. 用 `torch.gather` 取出实际下一个 token 的 log 概率——这就是 per_token_log_probs
> 4. 关键步骤：用 prompt_length 构造 response_mask，**把 prompt 位置的 log 概率清零**
> 5. 求和得到 response 部分的总 log 概率
>
> 为什么第 4 步这么重要？假设 prompt 有 50 个 token，chosen response 有 30 个 token，rejected response 有 40 个 token。如果不 mask 掉 prompt，那 log-prob 的差异会被 50 个 prompt token 的随机波动淹没——chosen 和 rejected 的 prompt 完全一样，它们的 prompt log-prob 差异纯粹是数值误差。
>
> **第三块：compute_dpo_loss**（核心 4 步）
> 1. 用 policy model 算 chosen 和 rejected 的 log-prob
> 2. 用 reference model 算 chosen 和 rejected 的 log-prob（torch.no_grad()）
> 3. 算 log ratio：policy_logp - ref_logp，分别对 chosen 和 rejected
> 4. DPO loss = -F.logsigmoid(β * (chosen_log_ratio - rejected_log_ratio)).mean()
>
> 同时还记录了 chosen_reward 和 rejected_reward，用于训练监控。reward = β * log_ratio。当 chosen_reward > rejected_reward（margin > 0），说明模型在正确方向上学习。
>
> **第四块：train_epoch + evaluate + train**
> 标准的训练循环：前向传播 -> loss -> 反向传播 -> 梯度裁剪（max_grad_norm=1.0）-> 优化器更新。验证时用 torch.no_grad()。train() 方法支持 max_steps 参数——我们会用它限制训练步数来加速演示。

👀 输出要点
- Cell 10 公式：log_ratio = log π(response) - log π_ref(response)
- Cell 11 DPOTrainer 关键参数：beta=0.5, lr=3e-6, weight_decay=0.01, max_grad_norm=1.0
- compute_log_probs：使用 prompt_length 构造 response_mask，只统计 response 部分
- compute_dpo_loss：4 步核心计算
- 训练监控指标：train_losses, val_losses, chosen_rewards, rejected_rewards

❓ 预判问题

Q: 为什么 DPO 的学习率（3e-6）比 SFT 小这么多？
A: DPO 是在已经对齐格式的 SFT 模型上做偏好调整，改变量应该很小。学习率太大会导致模型偏离参考模型太远，生成质量急剧下降（所谓「模式坍缩」）。3e-6 到 5e-6 是 DPO 常用范围。

Q: β=0.5 算大还是小？
A: 偏大。Ch10 用的是 β=0.1。β 越大，对偏离参考模型的惩罚越重，模型变化越小。0.5 是一个保守选择，适合小模型防止过拟合。生产中通常从 0.1 开始调。

Q: 为什么用 logsigmoid 而不是 sigmoid + log？
A: 数值稳定性。当输入是大负数时，sigmoid 接近 0，再取 log 会得到 -inf。F.logsigmoid 内部做了数值稳定的实现，避免了这个问题。

Q: response_mask 里的 `-1` 偏移是怎么回事？
A: 因为 log-prob 计算用了右移标签（labels = input_ids[:, 1:]），整个序列右移了一位。所以 response 的起始位置也要减 1：`start = prompt_length - 1`。

➡️ 转场

> Trainer 代码讲完了。下面加载 SFT 模型，开始训练！

---

## 第六段：执行 DPO 训练（Cell 12-15）

📍 浏览 Cell 12（Markdown 标题）、运行 Cell 13（加载 SFT 模型 + 参考模型）、运行 Cell 14（创建 Trainer + 执行训练）、运行 Cell 15（训练曲线可视化）

⏱ 时间分配：10 分钟

🎯 本段目标
- 理解双模型加载：policy 和 reference 都从 SFT checkpoint 初始化
- 观察训练日志中的 loss 和 reward_margin 变化
- 通过三张可视化图判断训练是否正常

🗣 讲课话术

> 运行 Cell 13。（运行 Cell 13）两个模型都从 SFT checkpoint 加载——model 是要训练的 policy，ref_model 是冻结的参考。参数量 **14.31M**。注意 ref_model 用的是同一个 checkpoint 的独立副本，不是共享引用。
>
> 运行 Cell 14。（运行 Cell 14）看训练配置——β=0.5，学习率 3e-6，max_steps=30。我们只跑 30 步来快速演示，正式训练会跑完整个 epoch。
>
> 看训练日志输出——
> - 模型参数: 14.31M
> - 训练样本: 8000
> - 最大步数: 30
>
> 训练结束后看关键数字：
> - 训练 Loss: **0.0018**——非常低，说明模型在这 30 步里已经强烈偏好 chosen
> - 验证 Loss: **0.0000**——验证集上 loss 接近零
> - 训练 Reward 边际: **0.1940**——正值，方向正确
> - 验证 Reward 边际: **20.9926**——非常大的正值，模型在验证集上已经极度偏好 chosen
>
> 模型保存成功：✓ 保存最佳模型（保存到 dpo_model 目录）
>
> 运行 Cell 15 看可视化。（运行 Cell 15）三张图：
> - 左图「DPO 训练损失」：Loss 快速下降
> - 中图「Chosen vs Rejected 奖励」：绿色（Chosen）在上方，红色（Rejected）在下方，两者分离
> - 右图「奖励间隔」：Margin 持续为正且递增
>
> 最终 Reward 边际输出的注释说得好：「正值表示模型更偏好 chosen 回答」。

👀 输出要点
- Cell 13：模型参数 14.31M，两个模型从 sft_model 加载
- Cell 14 训练结果：
  - 训练 Loss: 0.0018
  - 验证 Loss: 0.0000
  - 训练 Reward 边际: 0.1940
  - 验证 Reward 边际: 20.9926
  - 保存最佳模型到 dpo_model
- Cell 15 三张图：Loss 下降、Chosen/Rejected 奖励分离、Margin 递增为正

❓ 预判问题

Q: 验证 Reward 边际 20.99 会不会太大了？是不是过拟合？
A: 因为我们的训练集和验证集的偏好模式一致（都是结构化 vs 非结构化），而且只训练了 30 步，模型已经捕捉到了这个明显的格式差异。这不算过拟合，而是任务本身比较简单——结构化格式的信号太强了。

Q: 为什么只训 30 步？
A: 演示目的。8000 条数据 batch_size=4 一个 epoch 是 2000 步，跑完需要较长时间。30 步足以看到 loss 下降和 margin 变正的趋势。实际训练应该至少跑 1-3 个 epoch。

Q: 训练 Loss 0.0018 接近零是好事吗？
A: 说明模型在训练批次上非常确信 chosen 比 rejected 好。但要注意——如果 loss 在训练初期就接近零，可能说明 β 太小或数据太简单。我们的情况是 β=0.5 加上结构化 vs 非结构化的明显差异，loss 下降快是合理的。

➡️ 转场

> 数字上看训练成功了。但最终检验是——模型生成的回答真的变了吗？来做 SFT vs DPO 的对比测试。

---

## 第七段：SFT vs DPO 对比测试（Cell 16-20）

📍 浏览 Cell 16（Markdown 标题）、运行 Cell 17（chat 函数定义）、运行 Cell 18（加载 DPO 和 SFT 模型）、运行 Cell 19（SFT vs DPO 对比生成）、浏览 Cell 20（对比说明 Markdown）

⏱ 时间分配：8 分钟

🎯 本段目标
- 直观看到 SFT 模型和 DPO 模型的输出对比
- 理解小模型 + 字符级分词的固有局限
- 区分「概率偏好」和「生成质量」两个不同维度的评估

🗣 讲课话术

> Cell 17 定义了 chat 函数——标准的生成管道：格式化 prompt -> 编码 -> model.generate -> 解码 -> 截取 assistant 回复。temperature=0.7，top_k=50。
>
> 运行 Cell 18 加载两个模型。（运行 Cell 18）DPO 模型和 SFT 模型都加载成功。
>
> 运行 Cell 19——这是最有看头的部分。（运行 Cell 19）我们用训练集前 3 条做对比：
>
> 第一题「用简洁语言介绍Temperature」：
> - 期望的 chosen: 「要点：1) 定义：控制采样随机性的参数 2) 作用：数值越高越发散。」
> - SFT 模型输出: 「Tokenizer把文本切分成模型可处理的token。」
> - DPO 模型输出: 「Tokenizer把文本切分成模型可处理的token。」
>
> 等等——两个模型输出居然一样？而且都答偏了？别慌，Cell 20 解释了原因。
>
> Cell 20 写得很坦诚：模型只有 **14M 参数**、**381 词表**（字符级分词），生成泛化能力有限。即使是 SFT 模型也可能对训练集以外的指令给出无关回答。这是「小模型 + 字符级分词」的固有局限。
>
> 那 DPO 的效果体现在哪里？**体现在偏好概率上**！回看刚才的训练结果——Reward Margin > 0，100% 的情况下模型给 chosen 更高的概率。模型内部确实学会了「要点格式更好」，只是它的生成能力不足以可靠地产出这种格式。
>
> 这就好比一个小孩能分辨出哪幅画更好看（偏好），但自己画不出好看的画（生成）。偏好学习和生成能力是两个维度。
>
> Cell 20 最后指出：更直观的工业级效果对比请看 Ch10_DPO（用 Qwen2.5-0.5B），下一步的 05_Evaluation 会通过困惑度、偏好准确率等定量指标系统对比 Base/SFT/DPO 三阶段模型。

👀 输出要点
- Cell 18：两个模型加载成功
- Cell 19 对比结果：
  - 「用简洁语言介绍Temperature」—— SFT 和 DPO 输出相似，均有偏题现象
  - 「什么是BLEU？」—— SFT: DPO相关回答, DPO: Embedding相关回答
  - 「什么是训练集？」—— 两者输出相似
- Cell 20 说明：小模型局限，DPO 效果体现在偏好概率而非自由生成质量

❓ 预判问题

Q: SFT 和 DPO 输出几乎一样，说明 DPO 没用吗？
A: 不是。DPO 改变的是模型内部的概率分布——给结构化回答更高概率、给非结构化回答更低概率。但 14M 模型的生成能力不足以稳定产出高质量输出。就像调整了方向盘角度，但引擎马力不够——方向对了但跑不远。用更大的模型（如 Ch10 的 0.5B）效果会很明显。

Q: 模型回答的内容跟问题完全不相关怎么办？
A: 这是 14M 小模型 + 381 词表的固有局限。模型学到的知识有限，很多问题超出了它的能力范围。SFT 让它学会了格式（`<|system|>...<|assistant|>...`），但具体内容的准确性需要更大的模型和更多的预训练数据。

Q: 05_Evaluation 会怎么评估？
A: 用定量指标——困惑度（模型对 chosen 的困惑度应该比 rejected 低）、偏好准确率（模型给 chosen 的概率是否高于 rejected）。这些指标不依赖生成质量，直接衡量模型内部的偏好学习效果。

➡️ 转场

> 对比看完了。最后做个总结。

---

## 第八段：总结 + 下一步（Cell 21）

📍 浏览 Cell 21（总结 Markdown）

⏱ 时间分配：2 分钟

🎯 本段目标
- 串联本课四大模块
- 预告 Part 5 评估

🗣 讲课话术

> Cell 21 总结了三个部分：
>
> 第一，DPO 偏好数据——(prompt, chosen, rejected) 三元组，我们用了 8000 条训练集，偏好结构化「要点」风格。
>
> 第二，DPO 算法实现——参考模型计算基准概率，优化 log ratio 差异，β=0.5 控制偏离程度。
>
> 第三，训练监控——Loss、Chosen/Rejected Reward、Reward 边际。我们只跑了 30 步就看到了明确的偏好信号。
>
> 最后看 Cell 21 的 DPO vs RLHF 对比表：复杂度——RLHF 需要 Reward Model + PPO，DPO 直接优化；稳定性——PPO 难调参，DPO 更稳定；计算量——RLHF 大，DPO 小；效果——接近。
>
> 下一步是 05_Evaluation，我们会用困惑度、偏好准确率、生成质量三个维度系统对比 Base/SFT/DPO 三阶段模型。今天的内容就到这里！

👀 输出要点
- Cell 21 总结：偏好数据、DPO 实现、训练监控
- DPO vs RLHF 对比表
- 下一步：05_Evaluation（困惑度、偏好准确率、生成质量）

❓ 预判问题

Q: 如果要在自己的任务上用 DPO，应该怎么准备数据？
A: 三步：(1) 确定偏好维度（简洁性、安全性、专业性等）；(2) 对同一个 prompt 生成多个回答（用模型采样或人工编写）；(3) 人工标注或 AI 辅助标注哪个更好。数据质量决定上限。

Q: 这个 14M 模型的 DPO 结果能推广到大模型吗？
A: 原理完全一样，大模型效果会好得多。Ch10 用 Qwen2.5-0.5B（35 倍大）做了同样的 DPO 训练，回复长度减少明显、偏好准确率大幅提升。

---

## 附录

### 时间表汇总

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 0-10 min | 开场 + DPO 原理 + RLHF 对比 | Cell 0-1 | 10 min |
| 10-15 min | 环境设置 | Cell 2-3 | 5 min |
| 15-27 min | 偏好数据加载与分析 | Cell 4-5 | 12 min |
| 27-40 min | DPO 数据集实现 | Cell 6-9 | 13 min |
| 40-45 min | 休息 + 回顾 | — | 5 min |
| 45-60 min | DPO Trainer 核心实现 | Cell 10-11 | 15 min |
| 60-70 min | 执行训练 + 可视化 | Cell 12-15 | 10 min |
| 70-78 min | SFT vs DPO 对比测试 | Cell 16-20 | 8 min |
| 78-80 min | 总结 + 下一步 | Cell 21 | 2 min |

### 关键数据速查

| 数据项 | 值 |
|:---|:---|
| 偏好训练集 | 8000 条 |
| 偏好验证集 | 1000 条 |
| 偏好测试集 | 1000 条 |
| 偏好策略 | 结构化「要点」风格 vs 非结构化叙述 |
| 示例 chosen | 「要点：1) 定义：控制采样随机性的参数 2) 作用：数值越高越发散。」 |
| 示例 rejected | 「Temperature是控制采样随机性的参数，主要用于数值越高越发散。」 |
| 模型 | CustomGPT (14.31M 参数) |
| 词表大小 | 381（字符级分词） |
| MAX_LENGTH | 256 |
| BATCH_SIZE | 4 |
| beta (β) | 0.5 |
| 学习率 | 3e-6 |
| weight_decay | 0.01 |
| max_grad_norm | 1.0 |
| 训练 Epochs | 1 (max_steps=30) |
| 训练 Loss（最终） | 0.0018 |
| 验证 Loss（最终） | 0.0000 |
| 训练 Reward 边际 | 0.1940 |
| 验证 Reward 边际 | 20.9926 |
| 参考模型 | SFT 模型（冻结） |

### 应急预案

| 场景 | 应对 |
|:---|:---|
| SFT 模型不存在 | Cell 13 有 fallback——自动创建新模型（d_model=384, n_heads=6, n_layers=6）。提醒学生效果会不如 SFT 基础 |
| tokenizer 不存在 | Cell 9 有 fallback——用 DPO 数据现场构建词表 |
| GPU 显存不足 | 减小 MAX_LENGTH 至 128，BATCH_SIZE 至 2 |
| CPU 训练太慢 | 减小 max_steps 至 10；跳过 Cell 15 可视化直接看数字 |
| 训练 Loss 不下降 | 检查 ref_model 是否正确冻结（requires_grad=False）；检查 prompt_length 计算是否正确；尝试增大学习率至 1e-5 |
| SFT vs DPO 输出一样 | 正常现象——Cell 20 已解释。引导学生关注偏好概率（Reward Margin）而非生成内容。推荐对照 Ch10 看大模型效果 |
| 数据文件加载失败 | 检查 data/ 目录路径。Cell 5 的 resolve_data_dir() 会尝试当前目录和上级目录 |
| 学生问「DPO 没效果」 | 区分两个维度：(1) 偏好概率——Reward Margin > 0 说明有效；(2) 生成质量——受限于模型规模。偏好学习成功 ≠ 生成能力提升 |
| 时间不够 | 优先保证 Cell 0-1（原理）+ Cell 11（DPO Trainer）+ Cell 14（训练）+ Cell 19（对比）。其余可快速带过 |